# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer: Customer RFM Segmentation (`gold.customer_rfm_segments`)
# MAGIC 
# MAGIC - **Sources**: `silver.fct_orders_fulfillment`, `silver.dim_customers_scd2`
# MAGIC - **Target**: `gold.customer_rfm_segments` (Unity Catalog managed Delta table)
# MAGIC - **Key Analytics**:
# MAGIC   - Computes **Recency** (days since last order), **Frequency** (total orders placed), and **Monetary** (total spent).
# MAGIC   - Calculates percentile rankings (NTILE 1-4) across R, F, and M.
# MAGIC   - Derives actionable business segments: `Champions`, `Loyal Customers`, `Potential Loyalists`, `At Risk`, `Hibernating`.



In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer: Customer RFM Segmentation (`gold.customer_rfm_segments`)

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# COMMAND ----------

# 1. Read source data
df_fact = spark.table("silver.fct_orders_fulfillment")
df_cust = spark.table("silver.dim_customers_scd2").filter(F.col("is_current") == True)



In [0]:
# COMMAND ----------

# 2. Reference date for Recency
snapshot_date = df_fact.select(F.coalesce(F.max("order_date"), F.current_date())).collect()[0][0]

# COMMAND ----------

# 3. Aggregate Recency, Frequency, and Monetary metrics per user
# If total_amount was 0 due to unmapped product catalog prices, fall back to quantity-based unit pricing
df_rfm_base = (
    df_fact
    .filter(F.col("usr_id").isNotNull())
    .groupBy("usr_id")
    .agg(
        F.datediff(F.lit(snapshot_date), F.max("order_date")).alias("recency_days"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(
            F.sum(
                F.when(F.col("total_amount") > 0, F.col("total_amount"))
                .otherwise(F.col("quantity") * 25.0)  # Standard estimated item value fallback
            ), 
            2
        ).alias("monetary_value")
    )
)



In [0]:
# COMMAND ----------

# 4. Rank scores across percentiles
w_r = Window.orderBy(F.col("recency_days").desc())
w_f = Window.orderBy(F.col("frequency").asc())
w_m = Window.orderBy(F.col("monetary_value").asc())

df_rfm_scores = (
    df_rfm_base
    .withColumn("r_score", F.ntile(4).over(w_r))
    .withColumn("f_score", F.ntile(4).over(w_f))
    .withColumn("m_score", F.ntile(4).over(w_m))
    .withColumn("rfm_score", F.concat(F.col("r_score"), F.col("f_score"), F.col("m_score")))
)



In [0]:
# COMMAND ----------

# 5. Business Segmentation Logic
df_rfm_segmented = (
    df_rfm_scores
    .withColumn(
        "customer_segment",
        F.when((F.col("r_score") >= 3) & (F.col("f_score") >= 3) & (F.col("m_score") >= 3), F.lit("Champions"))
        .when((F.col("r_score") >= 3) & (F.col("f_score") >= 2), F.lit("Loyal Customers"))
        .when((F.col("r_score") >= 3) & (F.col("f_score") == 1), F.lit("Recent Customers"))
        .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), F.lit("At Risk Customers"))
        .when((F.col("r_score") == 1) & (F.col("f_score") == 1), F.lit("Lost / Hibernating"))
        .otherwise(F.lit("Potential Loyalists"))
    )
)



In [0]:
# COMMAND ----------

# 6. Join Customer Demographics
df_gold_rfm = (
    df_rfm_segmented.join(
        df_cust.select("usr_id", "customer_sk", "usr_city", "usr_state"),
        on="usr_id",
        how="left"
    )
    .withColumn("_transformed_at", F.current_timestamp())
    .select(
        F.coalesce(F.col("customer_sk"), F.lit("UNKNOWN")).alias("customer_sk"),
        F.col("usr_id"),
        F.coalesce(F.col("usr_city"), F.lit("Unknown")).alias("usr_city"),
        F.coalesce(F.col("usr_state"), F.lit("Unknown")).alias("usr_state"),
        F.col("recency_days"),
        F.col("frequency"),
        F.col("monetary_value"),
        F.col("r_score"),
        F.col("f_score"),
        F.col("m_score"),
        F.col("rfm_score"),
        F.col("customer_segment"),
        F.col("_transformed_at")
    )
)



In [0]:
# COMMAND ----------

# 7. Write to Unity Catalog
(
    df_gold_rfm.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.customer_rfm_segments")
)

spark.sql("OPTIMIZE gold.customer_rfm_segments ZORDER BY (customer_segment, usr_city)")

# COMMAND ----------

# 8. Display segment summary
display(
    spark.table("gold.customer_rfm_segments")
    .groupBy("customer_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("monetary_value"), 2).alias("avg_spent"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days")
    )
    .orderBy(F.col("customer_count").desc())
)